# 🤖 实验 22：大模型工具调用之层级代理 (Hierarchical Agent) 实战

在前面的实验中，我们的工具都是常规的 Python 函数、系统 CLI 命令行、或者 MCP 服务器。但是在复杂的智能体系统中，**大模型本身也可以作为另一个大模型的工具来使用**。这就是**层级代理 (Hierarchical Agents)** 架构。

### 💡 为什么需要将 LLM 作为工具？
1. **职责分离 (Specialization)**：主大模型（Orchestrator，类似于项目经理）只负责拆解规划任务与结果汇总。具体的子任务（如专业翻译、代码审计、文件浏览）通过工具调用分摊给配置了特定 System Prompt 角色定义和拥有独立工具集的“专职子大模型”（Worker Agent）。
2. **上下文隔离**：通过子 Agent 处理专项内容，可以大幅降低主大模型的上下文窗口负担，防止由于大量历史对话导致主模型注意力消散。

本实验中，我们将演示：主大模型（经理）收到“寻找 `target_simple.txt` 文件并将其内容翻译成英文”的任务，它会将“寻找文件”这一具体职责，通过调用“文件检索 Agent（员工）”工具来委派完成。员工 Agent 自己拥有寻找文件的 Function Calling 工具集，在得到文件内容后反馈给经理，经理再亲自把其内容翻译为英文输出给用户。

## 🛠️ 环境初始化：API 密钥与 OpenAI 客户端配置

为了运行本 Notebook，我们需要配置大模型 API。请在下方填入您的 API Key 和 Base URL。本代码默认支持通义千问 (DashScope) 与硅基流动 (SiliconFlow)，也可以直接使用 OpenAI 或其他兼容的 API 接口。

In [ ]:
import os
import json
import time
import httpx
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（也可以直接读取系统环境变量）
# ============================================================
API_KEY = ""         # 例如: "sk-abc123..."
BASE_URL = "https://api.siliconflow.cn/v1"        # 例如: "https://api.siliconflow.cn/v1" 或 "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V3"      # 例如: "deepseek-ai/DeepSeek-V3" 或 "qwen-plus"

# 优先从上面填写的变量读取，其次读取系统环境变量
api_key = API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("SILICONFLOW_API_KEY") or os.environ.get("DASH_SCOPE_API_KEY")
base_url = BASE_URL or os.environ.get("OPENAI_API_BASE")
model_name = MODEL_NAME or os.environ.get("OPENAI_API_MODEL")

# 自动识别常见服务商的环境变量
if not base_url:
    if os.environ.get("SILICONFLOW_API_KEY"):
        base_url = "https://api.siliconflow.cn/v1"
        model_name = model_name or "deepseek-ai/DeepSeek-V3"
    elif os.environ.get("DASH_SCOPE_API_KEY"):
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        model_name = model_name or "qwen-plus"
    else:
        base_url = "https://api.openai.com/v1"
        model_name = model_name or "gpt-4o-mini"

if not api_key:
    raise ValueError("❌ 未检测到 API Key！请在上方单元格中配置您的 API Key 或设置系统环境变量。")

# 💡 如果您在 Windows 环境下使用 VPN 或代理工具，可能会遇到 SSL/ConnectError 报错。
# 此时可以尝试取消下方这行代码的注释，使用禁用了 SSL 证书验证的自定义 httpx 客户端：
# openai_client = OpenAI(api_key=api_key, base_url=base_url, http_client=httpx.Client(verify=False))
openai_client = OpenAI(api_key=api_key, base_url=base_url)
print(f"✨ 客户端实例化成功！当前使用的接口地址为: {base_url}，模型为: {model_name}")

## 📂 测试数据生成

为了进行文件查找实验，我们需要在本地创建一个测试目录 `data/fc_test`，并在其中放入几个测试文件，包括我们要让大模型查找的目标文件 `target_simple.txt`。

In [ ]:
import shutil

# 创建测试目录
test_dir = "data/fc_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir, exist_ok=True)

# 写入一些干扰文件与目标文件
files_to_create = {
    "report.txt": "2026年年度财务报告：公司业绩稳步上升。",
    "config.json": '{"port": 8080, "debug": false}',
    "target_simple.txt": "恭喜你！成功找到了 target_simple.txt 文件。密钥为: SIMPLE_SUCCESS_2026",
    "notes.log": "2026-06-03 12:00:00 - Server started successfully."
}

for filename, content in files_to_create.items():
    with open(os.path.join(test_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"💾 测试目录 {test_dir} 准备完毕，已创建 {len(files_to_create)} 个文件。")

---
## 🧱 第一步：定义底层文件操作函数

这些函数是最终去执行物理文件读取的 Python 函数，也是员工 Agent（File Explorer Agent）拥有的工具库。

In [ ]:
def list_top_level_files() -> list:
    """列出 data/fc_test 目录下的所有文件名。"""
    base_dir = "data/fc_test"
    if not os.path.exists(base_dir):
        return []
    return [f for f in os.listdir(base_dir) if os.path.isfile(os.path.join(base_dir, f))]

def check_file_exists(filename: str) -> str:
    """在本地 data/fc_test 目录下检查指定的文件是否存在，如果存在则读取其内容并返回。"""
    base_dir = "data/fc_test"
    filepath = os.path.join(base_dir, filename)
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    return f"文件 {filename} 不存在于 {base_dir} 中。"

print("✅ 基础文件工具函数就绪。")

## 👤 第二步：实现“员工”智能体 (File Explorer Agent) 并包装成工具

我们开发一个普通的 Python 函数 `ask_file_agent(query: str) -> str`。在这个函数内部，我们会用特定的 **System Prompt**（定义其为专业文件检索员）和**独立的 tools** 来启动一个完整的 LLM 交互会话，直至其获取到文件数据。这样，这个函数对外就表现为一个可以被经理大模型直接调用的“工具”。

In [ ]:
def ask_file_agent(query: str) -> str:
    """
    文件检索专员智能体。他配置有专用的 System Prompt 并拥有文件查找工具。
    主模型在需要寻找文件时可以向他提问。
    """
    print(f"\n👤 [经理 ➔ 专员] 派发工作指令: '{query}'")
    
    worker_system_prompt = (
        "你是一个专门在本地 data/fc_test 目录下寻找文件的辅助智能体。\n"
        "你拥有 list_top_level_files 和 check_file_exists 两个文件操作工具。\n"
        "你应该先调用列出文件工具，确认目标是否存在，然后再读取其具体内容。\n"
        "任务完成后，只需给出简洁的内容报告，不要带有多余的问候语或解释。"
    )
    
    worker_tools = [
        {
            "type": "function",
            "function": {
                "name": "list_top_level_files",
                "description": "列出 data/fc_test 目录下的所有文件名。"
            }
        },
        {
            "type": "function",
            "function": {
                "name": "check_file_exists",
                "description": "在本地 data/fc_test 目录下检查指定的文件是否存在，如果存在则读取其内容并返回。",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "filename": {
                            "type": "string",
                            "description": "要检查的文件名，例如 'target_simple.txt'"
                        }
                    },
                    "required": ["filename"]
                }
            }
        }
    ]
    
    worker_messages = [
        {"role": "system", "content": worker_system_prompt},
        {"role": "user", "content": query}
    ]
    
    max_turns = 5
    turn = 0
    
    # 启动子 Agent 交互循环 (使用 while 循环支持多步工具调用)
    while turn < max_turns:
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=worker_messages,
            tools=worker_tools,
            tool_choice="auto"
        )
        
        assistant_msg = response.choices[0].message
        worker_messages.append(assistant_msg)
        tool_calls = assistant_msg.tool_calls
        
        # 如果不需要再调用工具，说明最终结论已经生成，返回之
        if not tool_calls:
            return assistant_msg.content.strip()
            
        # 处理这一轮的工具调用
        for tool_call in tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"   🔧 [专员思考并调用工具] {func_name} | 参数: {func_args}")
            
            if func_name == "list_top_level_files":
                res = str(list_top_level_files())
            elif func_name == "check_file_exists":
                res = check_file_exists(func_args.get("filename"))
            else:
                res = "Error: Unknown tool"
                
            print(f"   ▶️ [专员工具执行反馈]: '{res}'")
            
            worker_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": res
            })
            
        turn += 1
        
    return "Error: Agent reached maximum turns without completing task."

# 验证“员工”智能体的工作效果
print("🧪 验证专员工作情况：")
print("专员最终答复:", ask_file_agent("查找名为 target_simple.txt 的文件，并输出文件中的内容。"))

## 👑 第三步：编写主智能体 (Orchestrator) 决策逻辑

现在我们注册一个给主大模型看懂的工具描述。主大模型只拥有一个叫做 `ask_file_agent` 的专员调用通道。我们给经理提问：“查找 `target_simple.txt` 文件并将其内容翻译为英文”。我们观察经理是如何通过派发任务给专员，并最终自行翻译结果的。

In [ ]:
orchestrator_tools = [
    {
        "type": "function",
        "function": {
            "name": "ask_file_agent",
            "description": "文件检索专员。当需要获取 data/fc_test 目录下的文件清单、查询某个文件的存在性或获取某文件的具体内容时，必须调用此工具委派任务。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "具体交代给文件检索专员的任务问题，例如：'读取target_simple.txt文件的内容'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# 经理收到用户的综合诉求：要求查找到文件，并且把里面的中文翻译为英文
user_target = "请帮我查找 data/fc_test 目录下的 target_simple.txt 文件。如果存在，请读取其内容并翻译成英文告诉我。"
print(f"👤 用户原始指令: '{user_target}'\n")

manager_messages = [{"role": "user", "content": user_target}]

# 第一阶段：经理进行顶层规划，决定委派任务
response = openai_client.chat.completions.create(
    model=model_name,
    messages=manager_messages,
    tools=orchestrator_tools,
    tool_choice="auto"
)

manager_message = response.choices[0].message
manager_calls = manager_message.tool_calls
manager_messages.append(manager_message)

if manager_calls:
    print("👑 [经理规划成功] 决策结果: 【需要向专员提问】")
    for call in manager_calls:
        func_name = call.function.name
        func_args = json.loads(call.function.arguments)
        
        if func_name == "ask_file_agent":
            # 委派工作给专员子 Agent
            task_query = func_args.get("query")
            agent_reply = ask_file_agent(task_query)
            print(f"\n📥 [经理收到专员返回的工作报告]:\n{agent_reply}")
            
            # 将报告作为执行结果传回上下文
            manager_messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": func_name,
                "content": agent_reply
            })
            
    # 第二阶段：经理汇总并做后期加工（如语言翻译）
    print("\n👑 经理正在根据报告整理并执行最终翻译...")
    final_response = openai_client.chat.completions.create(
        model=model_name,
        messages=manager_messages
    )
    
    print("\n🤖 经理交付给用户的最终总结答案:")
    print("=" * 65)
    print(final_response.choices[0].message.content)
    print("=" * 65)
else:
    print("经理判定无需使用专员，直接回复:")
    print(manager_message.content)

## 📈 课后知识复盘与思考练习

### 🧠 思考题
1. 层级代理（Hierarchical Agents）和多工具 Function Calling（一个大模型同时加载 20 个工具）相比，在系统架构、Prompt 编写的复杂度、以及长上下文维护效率上，各自有什么优势和弊端？